# 04 - Baselines

Two reference models that learn nothing from our data. Both reduce each scalogram to 16 numbers with PCA, the same size as the VAE latent vector, and then use the same GMM step as the VAE notebooks, including the daily price level appended to the 16 numbers.

1. Raw PCA: PCA directly on the 3,072 pixels of each scalogram. Answers whether any neural network helps at all.
2. MobileNetV2 PCA: a MobileNetV2 trained on ImageNet photos, with its weights frozen and its classifier removed, gives 1,280 features per scalogram (as in Benavides and Luna, 2024). PCA reduces them to 16.

PCA and the GMM are fitted on the training days only. The last section compares every model that has been run.

<d.benavidess@uniandes.edu.co>

In [ ]:
import sys
sys.path.append("..")

import json
from pathlib import Path

import torch

import numpy as np
import pandas as pd

import altair as alt
import matplotlib.pyplot as plt
from prettytable import PrettyTable
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

plt.rcParams['font.family'] = 'Arial'
alt.data_transformers.enable('vegafusion')

from src.data import load_scalograms, load_context, chronological_split
from src.models.baselines import raw_features, mobilenet_features, pca_reduce
from src.training import set_seed
from src.clustering import with_level, assign_regimes, summary_metrics, regime_profile, save_regimes
from src.plots import selection_chart, regime_timeline

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## 0. Experiment Run

In [ ]:
RUN_NAME = "run01_k8"

## 1. Configuration

In [ ]:
RUN_NAME = "run01_k4_shape"

SCALOGRAM_DIR = Path("../data/processed/scalograms")
DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models/baselines") / RUN_NAME
FIGURE_DIR = Path("../figures/baselines") / RUN_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "n_components": 16,
    "image_size": 224,
    "batch_size": 64,
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "n_regimes": 4,
    "k_range": [2, 3, 4, 5, 6, 7, 8],
    "seed": 42,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(CONFIG["seed"])
(MODEL_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
print(f"Device: {DEVICE}")

## 2. Data

In [ ]:
images, metadata = load_scalograms(SCALOGRAM_DIR)
context = load_context(DATA_DIR, metadata)
train_idx, val_idx, test_idx = chronological_split(len(images), CONFIG["train_ratio"], CONFIG["val_ratio"])
split_dates = [context["date"].iloc[val_idx[0]], context["date"].iloc[test_idx[0]]]

IMG_C, IMG_H, IMG_W = images.shape[1:]
print(f"Scalograms: {images.shape}, range [{images.min():.3f}, {images.max():.3f}]")

t = PrettyTable(["Split", "Days", "From", "To"])
for name, idx in (("train", train_idx), ("val", val_idx), ("test", test_idx)):
    t.add_row([name, len(idx), context["date"].iloc[idx[0]].date(), context["date"].iloc[idx[-1]].date()])
print(t)

In [ ]:
def run_baseline(name, latents):
    model_dir = MODEL_DIR / name
    figure_dir = FIGURE_DIR / name
    figure_dir.mkdir(parents=True, exist_ok=True)
    features = with_level(latents, metadata, train_idx)

    selection, k, labels, probs = assign_regimes(
        features, train_idx, context["precio_real_cop_kwh"].to_numpy(),
        k=CONFIG["n_regimes"], k_range=CONFIG["k_range"], seed=CONFIG["seed"],
    )
    ctx = context.assign(regime=labels)
    metrics = {"model": name, "k": k, **summary_metrics(features, labels, seed=CONFIG["seed"])}
    save_regimes(model_dir, ctx, labels, probs, metrics)
    np.save(model_dir / "latents.npy", latents)

    selection_chart(selection, k).save(str(figure_dir / "gmm_selection.png"), scale_factor=2.0)
    timeline = regime_timeline(ctx, split_dates, title=f"{name}: detected regimes (k={k})")
    timeline.save(str(figure_dir / "regime_timeline.png"), scale_factor=2.0)

    t = PrettyTable(["Regime", "Days", "Share %", "Real price", "Reservoir", "Mean ONI", "El Nino %"])
    for _, row in regime_profile(ctx).iterrows():
        t.add_row([int(row["regime"]), int(row["days"]), f"{row['share_pct']:.1f}", f"{row['mean_real_price']:.1f}",
                   f"{row['mean_reservoir']:.3f}", f"{row['mean_oni']:.2f}", f"{row['el_nino_pct']:.1f}"])
    print(f"{name}: k = {k}")
    if CONFIG["n_regimes"] is None and k == max(CONFIG["k_range"]):
        print("BIC is still falling at the largest k. Choose k from the table and set CONFIG['n_regimes'] by hand.")
    print(t)
    return timeline

## 3. Raw PCA

In [ ]:
raw_latents, raw_var = pca_reduce(raw_features(images), train_idx, CONFIG["n_components"])
print(f"Explained variance with {CONFIG['n_components']} components: {raw_var.sum():.1%}")

In [ ]:
raw_timeline = run_baseline("raw_pca", raw_latents)
raw_timeline

## 4. MobileNetV2 PCA

The first run downloads the ImageNet weights (about 14 MB) and extracts features for all days. The features are cached in `models/baselines/<RUN_NAME>/` so later runs skip this step. Each scalogram is repeated to 3 channels, resized from 128 x 24 to 224 x 224 and normalized with ImageNet statistics.

In [ ]:
feature_path = MODEL_DIR / "mobilenet_features.npy"
if feature_path.exists():
    mobilenet_feats = np.load(feature_path)
else:
    mobilenet_feats = mobilenet_features(images, DEVICE, CONFIG["batch_size"], CONFIG["image_size"])
    np.save(feature_path, mobilenet_feats)

mobilenet_latents, mobilenet_var = pca_reduce(mobilenet_feats, train_idx, CONFIG["n_components"])
print(f"Features: {mobilenet_feats.shape}")
print(f"Explained variance with {CONFIG['n_components']} components: {mobilenet_var.sum():.1%}")

In [ ]:
mobilenet_timeline = run_baseline("mobilenet_pca", mobilenet_latents)
mobilenet_timeline

## 5. Comparison across models

Reads the metrics of every model that has been run. Runs marked "nominal, no demeaning" used the earlier scalograms and are kept as evidence of the change. There are no true regime labels, so the comparison uses cluster separation (silhouette, on each model's own latent space), regime persistence (run length, switches per year) and, in each notebook, the regime profile against price, reservoir level and ONI. Silhouette values from different latent spaces are only roughly comparable.

In [ ]:
runs = {
    "ConvVAE": Path("../models/vae/" + RUN_NAME + "/metrics.json"),
    "Temporal VAE": Path("../models/temporal_vae/" + RUN_NAME + "/metrics.json"),
    "Raw PCA": MODEL_DIR / "raw_pca" / "metrics.json",
    "MobileNetV2 PCA": MODEL_DIR / "mobilenet_pca" / "metrics.json",
    "ConvVAE (nominal, no demeaning)": Path("../models/vae/" + RUN_NAME + "/metrics.json"),
    "Raw PCA (nominal, no demeaning)": Path("../models/baselines/raw_pca/metrics.json"),
    "MobileNetV2 PCA (nominal, no demeaning)": Path("../models/baselines/mobilenet_pca/metrics.json"),
}

t = PrettyTable(["Model", "k", "Silhouette", "Mean run (days)", "Median run (days)", "Switches / year"])
for label, path in runs.items():
    if not path.exists():
        t.add_row([label, "not run", "", "", "", ""])
        continue
    m = json.loads(path.read_text())
    t.add_row([label, m["k"], f"{m['silhouette']:.3f}", f"{m['mean_run_days']:.1f}",
               f"{m['median_run_days']:.0f}", f"{m['switches_per_year']:.1f}"])
print(t)